## Scraping Daftar Kabkot di Tokped

In [ ]:
import requests
import json
import pandas as pd
import os

def get_kabkot_sumut():
    url = "https://gql.tokopedia.com/graphql/DynamicAttributes"
    
    # Gunakan Session untuk efisiensi koneksi
    session = requests.Session()
    
    headers = {
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Accept": "*/*",
        "Origin": "https://www.tokopedia.com",
        "Referer": "https://www.tokopedia.com/p/handphone-tablet/handphone",
        "X-Version": "864190c", # String versi GQL (sering berubah, tapi ini cukup stabil)
        "Sec-Fetch-Dest": "empty",
        "Sec-Fetch-Mode": "cors",
        "Sec-Fetch-Site": "same-site"
    }

    payload = [{
        "operationName": "DynamicAttributes",
        "variables": {"source": "directory", "q": "", "filter": {"sc": "2436"}},
        "query": """query DynamicAttributes($source: String, $q: String, $filter: DAFilterQueryType) {
          dynamicAttribute(source: $source, q: $q, filter: $filter) {
            data { filter { options { name key value child { name key value } } } }
          }
        }"""
    }]

    blacklist_ids = ["520"]

    try:
        print("Sedang mencoba menghubungi Tokopedia (Timeout dinaikkan ke 30 detik)...")
        # Timeout dinaikkan ke 30 agar tidak cepat putus
        response = session.post(url, headers=headers, json=payload, timeout=30)
        
        response.raise_for_status() # Cek apakah status code 200
        full_data = response.json()
        
        # Validasi data
        if not full_data or 'data' not in full_data[0]:
            return "Error: Respon kosong dari server. Coba lagi dalam 5 menit."

        dynamic_attr = full_data[0].get('data', {}).get('dynamicAttribute', {}).get('data', {})
        filters = dynamic_attr.get('filter', [])
        
        sumut_ids = []
        all_potential_cities = []

        for f in filters:
            for opt in f.get('options', []):
                all_potential_cities.append(opt)
                if opt.get('child'):
                    all_potential_cities.extend(opt['child'])
                if opt.get('name') == "Sumatera Utara":
                    sumut_ids = opt.get('value', '').split(',')

        hasil_kabkot = []
        seen_ids = set()
        
        for city in all_potential_cities:
            val = city.get('value')
            if val in sumut_ids and val not in blacklist_ids and val not in seen_ids:
                hasil_kabkot.append({
                    "id": val,
                    "nama": city.get('name')
                })
                seen_ids.add(val)

        return sorted(hasil_kabkot, key=lambda x: x['nama'])

    except requests.exceptions.ReadTimeout:
        return "Error: Timeout lagi. Coba gunakan VPN atau ganti koneksi internet (Hotspot HP biasanya lebih lancar)."
    except Exception as e:
        return f"Error: {str(e)}"

# --- Eksekusi tetap sama ---
kabkot = get_kabkot_sumut()

if isinstance(kabkot, list):
    print(f"Berhasil menarik {len(kabkot)} wilayah di Sumut (Pariaman dihapus).")
    
    # 1. Tampilkan di Console
    for item in kabkot:
        print(f"[{item['id']}] {item['nama']}")
    
    # 2. Simpan ke Excel/CSV menggunakan Pandas
    df_kabkot = pd.DataFrame(kabkot)
    df_kabkot.to_excel("daftar_kabkot_sumut.xlsx", index=False)
    df_kabkot.to_csv("daftar_kabkot_sumut.csv", index=False)
    
    print("\nFile 'daftar_kabkot_sumut.xlsx' dan '.csv' telah dibuat!")
else:
    print(kabkot)

## Scraping Daftar Kategori di Tokped

In [ ]:
import json
import re
import csv

def scrape_all_tokopedia_categories(html_file_path):
    try:
        with open(html_file_path, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"File {html_file_path} tidak ditemukan.")
        return

    # 1. Ekstrak JSON __cache
    match = re.search(r'window\.__cache\s*=\s*({.*?});', content, re.DOTALL)
    if not match:
        print("Gagal menemukan window.__cache.")
        return

    try:
        raw_data = json.loads(match.group(1), strict=False)
    except Exception as e:
        print(f"Gagal parse JSON: {e}")
        return

    # 2. Kumpulkan SEMUA kategori yang ada ke dalam satu map
    # Kita bersihkan key-nya agar hanya ID angka saja
    all_cats = {}
    for key, val in raw_data.items():
        if isinstance(val, dict) and val.get('__typename') == 'AllCategoryResp':
            cat_id = str(val.get('id'))
            if cat_id != "None":
                all_cats[cat_id] = val

    master_list = []

    # 3. Iterasi semua kategori untuk mencari Level 1 (Kategori Utama)
    # Ciri L1 di Tokped: URL-nya pendek (misal: /p/rumah-tangga)
    # Atau kita cari yang "punya anak" tapi "bukan anak dari siapa-siapa"
    
    # Kita cari manual L1 yang umum (Rumah Tangga, Fashion, dll)
    # Berdasarkan data kamu, ID 984 adalah Rumah Tangga (L1)
    for cid, cat in all_cats.items():
        # Kita hanya proses L1 di sini (yang punya 'identifier' biasanya L1)
        if cat.get('identifier') and cid != "0":
            l1 = cat
            l2_refs = l1.get('child', [])
            
            if not l2_refs:
                master_list.append(format_data(l1))
                continue

            for l2_ref in l2_refs:
                # Ambil ID asli dari AllCategoryRespXXXX
                l2_id = l2_ref['id'].replace("AllCategoryResp", "")
                l2 = all_cats.get(l2_id)
                if not l2: continue

                l3_refs = l2.get('child', [])
                if not l3_refs:
                    master_list.append(format_data(l1, l2))
                else:
                    for l3_ref in l3_refs:
                        l3_id = l3_ref['id'].replace("AllCategoryResp", "")
                        l3 = all_cats.get(l3_id)
                        if l3:
                            master_list.append(format_data(l1, l2, l3))

    # 4. Simpan ke CSV
    output_file = 'kategori_tokopedia_FULL.csv'
    headers = ['L1_Name', 'L2_Name', 'L3_Name', 'Category_ID', 'URL']
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=headers, delimiter=";")
        writer.writeheader()
        writer.writerows(master_list)

    print(f"Buset! Berhasil narik {len(master_list)} kategori/sub-kategori!")

def format_data(l1, l2=None, l3=None):
    target = l3 if l3 else (l2 if l2 else l1)
    return {
        'L1_Name': l1.get('name', ''),
        'L2_Name': l2.get('name', '') if l2 else '',
        'L3_Name': l3.get('name', '') if l3 else '',
        'Category_ID': target.get('id', ''),
        'URL': target.get('url', '')
    }


# Download HTML dari Tokopedia "https://www.tokopedia.com/p" (buka halaman kategori, lalu Save As > Webpage, Complete)
# Lalu jalankan fungsi ini dengan nama file HTML yang sudah kamu simpan tadi
scrape_all_tokopedia_categories('Daftar Produk _ Tokopedia.html')